# ECS 170 Stage 3 — CIFAR-10 CNN (ResNet-18 + MixUp)
**Runtime → Change runtime type → T4 GPU** before running.

Expected time: ~**45–55 minutes** for all 3 configs, 50 epochs each.  
Expected accuracy: **96–97%** (default config) with MixUp augmentation.


In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ── Cell 2: Imports ─────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import os
import time
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

os.makedirs('/content/results', exist_ok=True)
DEVICE = torch.device('cuda')
print('Ready.')

In [ ]:
# ── Cell 3: ResNet-18 (CIFAR adapted) ───────────────────────────

class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1,      padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = F.relu(out + self.shortcut(x))
        return out


class ResNet18_CIFAR(nn.Module):
    """ResNet-18 adapted for CIFAR-10 (32x32 input, no initial maxpool)."""
    def __init__(self, n_classes=10, dropout=0.0):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64,  64,  2, stride=1)
        self.layer2 = self._make_layer(64,  128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.drop   = nn.Dropout(dropout)
        self.fc     = nn.Linear(512, n_classes)
        # Kaiming init
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, in_ch, out_ch, n_blocks, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(1, n_blocks):
            layers.append(BasicBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.pool(x).view(x.size(0), -1)
        x = self.drop(x)
        return self.fc(x)

print('Model defined.')

In [ ]:
# ── Cell 4: Data loaders ─────────────────────────────────────────
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

def get_dataloaders(batch_size=256):
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10('/content/data', train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10('/content/data', train=False, download=True, transform=test_tf)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')
    return train_loader, test_loader

print('Data loader defined.')

In [ ]:
# Cell 5 — Train / evaluate helpers (with MixUp for 96%+ accuracy)

def mixup_batch(imgs, labels, alpha=0.2, n_classes=10, device='cuda'):
    """MixUp: randomly blend two images and their one-hot labels."""
    lam = float(torch.distributions.Beta(alpha, alpha).sample())
    idx = torch.randperm(imgs.size(0), device=device)
    mixed_imgs = lam * imgs + (1 - lam) * imgs[idx]
    # One-hot encode for mixed loss
    y1 = torch.nn.functional.one_hot(labels,          n_classes).float()
    y2 = torch.nn.functional.one_hot(labels[idx],     n_classes).float()
    mixed_labels = lam * y1 + (1 - lam) * y2
    return mixed_imgs, mixed_labels


def train_epoch(model, loader, optimizer, criterion, device, scheduler, use_mixup=True):
    """One full training epoch. scheduler.step() called per BATCH (required for OneCycleLR)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        if use_mixup:
            mixed_imgs, mixed_labels = mixup_batch(imgs, labels, alpha=0.2, device=device)
            out  = model(mixed_imgs)
            # Soft cross-entropy for mixed labels
            log_probs = torch.nn.functional.log_softmax(out, dim=1)
            loss = -(mixed_labels * log_probs).sum(dim=1).mean()
            # Accuracy: use original labels for readability
            correct += (out.argmax(1) == labels).sum().item()
        else:
            out  = model(imgs)
            loss = criterion(out, labels)
            correct += (out.argmax(1) == labels).sum().item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()          # OneCycleLR must step every batch
        total_loss += loss.item() * imgs.size(0)
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            preds = model(imgs.to(device)).argmax(1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return {
        'accuracy':           accuracy_score(labels, preds),
        'precision_macro':    precision_score(labels, preds, average='macro',    zero_division=0),
        'precision_weighted': precision_score(labels, preds, average='weighted', zero_division=0),
        'recall_macro':       recall_score(labels,   preds, average='macro',    zero_division=0),
        'recall_weighted':    recall_score(labels,   preds, average='weighted', zero_division=0),
        'f1_macro':           f1_score(labels,       preds, average='macro',    zero_division=0),
        'f1_weighted':        f1_score(labels,       preds, average='weighted', zero_division=0),
        'f1_micro':           f1_score(labels,       preds, average='micro',    zero_division=0),
    }

print('Helpers defined (MixUp enabled).')


## Run All 3 Configs

| Config | LR | Dropout | Notes |
|--------|----|---------|-------|
| `CIFAR_default` | 0.1 | 0.0 | Baseline — expect **93–95%** |
| `CIFAR_high_dropout` | 0.1 | 0.3 | Ablation: regularization |
| `CIFAR_low_lr` | 0.01 | 0.0 | Ablation: learning rate |

In [ ]:
# Cell 6 — Experiment runner

def run_experiment(config_name, max_epoch=30, lr=0.1, dropout=0.0, batch_size=256):
    print(f'\n{"="*60}')
    print(f'  Config : {config_name}')
    print(f'  epochs={max_epoch} | lr={lr} | dropout={dropout} | batch={batch_size}')
    print(f'{"="*60}')

    train_loader, test_loader = get_dataloaders(batch_size)
    model     = ResNet18_CIFAR(n_classes=10, dropout=dropout).to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr,
        momentum=0.9, nesterov=True, weight_decay=5e-4
    )
    # OneCycleLR: ramps up for 10% of total steps, then cosine anneals.
    # MUST step() once per batch — done inside train_epoch.
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        epochs=max_epoch,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy='cos',
        div_factor=10.0
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    loss_hist, acc_hist = [], []
    t0 = time.time()
    for epoch in range(1, max_epoch + 1):
        t_ep = time.time()
        loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scheduler)
        metrics  = evaluate(model, test_loader, DEVICE)
        test_acc = metrics['accuracy']
        loss_hist.append(loss)
        acc_hist.append(test_acc)
        print(f'  Ep {epoch:3d}/{max_epoch} | Loss: {loss:.4f} | '
              f'Train: {train_acc:.4f} | Test: {test_acc:.4f} | {time.time()-t_ep:.1f}s')

    total_time    = time.time() - t0
    final_metrics = evaluate(model, test_loader, DEVICE)

    out = {
        'config': config_name,
        'max_epoch': max_epoch, 'lr': lr, 'dropout': dropout,
        'training_time_sec': round(total_time, 1),
        'loss_history': loss_hist,
        'acc_history':  acc_hist,
        **final_metrics
    }
    with open(f'/content/results/{config_name}_metrics.json', 'w') as f:
        json.dump(out, f, indent=2)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(loss_hist); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax2.plot(acc_hist);  ax2.set_title('Test Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    fig.suptitle(config_name); plt.tight_layout()
    plt.savefig(f'/content/results/{config_name}_learning_curves.png', dpi=150)
    plt.close()

    print(f'  >>> DONE: Acc={final_metrics["accuracy"]:.4f} | '
          f'F1(macro)={final_metrics["f1_macro"]:.4f} | '
          f'Total: {total_time/60:.1f} min')
    return final_metrics

print('Runner defined.')


In [ ]:
# Cell 7 — Run all experiments (~50 min total on T4)
# MixUp + 50 epochs → targets 96–97% on CIFAR_default
results = {}
results['default']      = run_experiment('CIFAR_default',      max_epoch=50, lr=0.1,  dropout=0.0)
results['high_dropout'] = run_experiment('CIFAR_high_dropout', max_epoch=50, lr=0.1,  dropout=0.3)
results['low_lr']       = run_experiment('CIFAR_low_lr',       max_epoch=50, lr=0.01, dropout=0.0)


In [ ]:
# ── Cell 8: Summary table ────────────────────────────────────────
print('\n' + '='*70)
print('  CIFAR-10 FINAL SUMMARY')
print('='*70)
print(f'{"Config":<22} {"Acc":>7} {"Prec(M)":>9} {"Rec(M)":>8} {"F1(M)":>7} {"F1(W)":>7}')
print('-'*70)
for cfg, m in results.items():
    print(f'  {cfg:<20} {m["accuracy"]:>7.4f} '
          f'{m["precision_macro"]:>9.4f} '
          f'{m["recall_macro"]:>8.4f} '
          f'{m["f1_macro"]:>7.4f} '
          f'{m["f1_weighted"]:>7.4f}')
print('='*70)

In [ ]:
# ── Cell 9: Download result files ───────────────────────────────
import shutil
shutil.make_archive('/content/CIFAR_results', 'zip', '/content/results')

from google.colab import files
files.download('/content/CIFAR_results.zip')
print('Downloading CIFAR_results.zip ...')